# ForestWatch Papua — Banding 3 Model, Tetapkan Pemenang, Inferensi & 7 File Kontrak

**Jalankan di SATU komputer SETELAH ke-3 notebook training selesai & Drive ter-sync.**

Notebook ini: (1) baca `summary.json` ke-3 model dari
`ForestWatch_Outputs/Model_Comparison/`, (2) bandingkan akurasi (test mIoU pada
Papua holdout yang identik), (3) tetapkan **pemenang**, (4) promosikan artefak
pemenang ke lokasi kanonik (`ForestWatch_Patches/best_model.pt`, `ForestWatch_Outputs/`),
(5) inferensi T1+T2, (6) generate + validasi **7 file kontrak** untuk hilir (Orang 2 / WebGIS).

## Bagian 0 — Setup environment (Colab **atau** Komputer Lab)

Notebook ini **berdiri sendiri**. Ia **memuat hasil EDA & preprocessing**
(Bagian 1–14 dari `forestwatch_papua_full_pipeline.ipynb`) yang sudah tersimpan
di Google Drive — **tidak menghitung ulang**.

- **Google Colab** → set `ENV = "colab"`. Sel setup meng-clone repo, install
  package, lalu mount Drive.
- **Komputer lab** → set `ENV = "lab"`. Prasyarat **sekali saja**:
  1. Install **Google Drive for Desktop**, login akun yang sama, set folder
     `Satria Data 3.0` ke mode **Mirror** (bukan *Stream-only*) supaya file `.npz`
     benar-benar ada di disk lokal (DataLoader membaca ribuan file tiap epoch).
  2. Di clone repo lokal jalankan: `pip install -e ".[ml]"`.
  3. Sesuaikan `DRIVE_ROOT` ke path mount Drive Desktop (mis. `G:/My Drive/Satria Data 3.0`).

In [ ]:
# === Bagian 0 — Setup (set ENV = "colab" ATAU "lab") ===
ENV = "lab"   # ganti "colab" kalau jalan di Google Colab
from pathlib import Path

if ENV == "colab":
    import subprocess, sys, importlib
    # clone pertama kali / pull sesi berikutnya (selalu kode terbaru), lalu install.
    subprocess.run(
        "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
        "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
        shell=True, check=False,
    )
    subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
    if "/content/fw_repo/src" not in sys.path:
        sys.path.insert(0, "/content/fw_repo/src")
    for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN path mount Drive Desktop lab
else:
    raise ValueError("ENV harus 'colab' atau 'lab'")

assert DRIVE_ROOT.exists(), (
    f"DRIVE_ROOT {DRIVE_ROOT} tidak ada — cek mount Drive / sync (mode Mirror)."
)

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | DRIVE_ROOT={DRIVE_ROOT} | CUDA={torch.cuda.is_available()}{_gpu}")

In [ ]:
# === Deklarasi path gdrive (hasil EDA & preprocessing tersimpan di sini) ===
TILES_T1   = DRIVE_ROOT / 'ForestWatch_Tiles_T1'
TILES_T2   = DRIVE_ROOT / 'ForestWatch_Tiles_T2'
PATCH_DIR  = DRIVE_ROOT / 'ForestWatch_Patches'           # patch Papua (+ ckpt kanonik)
PATCHES_TRANSFER  = DRIVE_ROOT / 'ForestWatch_Patches_Transfer'
AUGMENTED_PATCHES = DRIVE_ROOT / 'Augmented_Patches'
DIST_DIR   = DRIVE_ROOT / 'Distribution_Reports'
MASK_DIR   = DRIVE_ROOT / 'ForestWatch_Masks'
OUT_DIR    = DRIVE_ROOT / 'ForestWatch_Outputs'
MODELS_ROOT = OUT_DIR / 'Model_Comparison'                # folder induk 3 model
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

from forestwatch.config import load_config
from forestwatch.utils.io import save_json, load_json
cfg = load_config()
print('Resep training :', cfg['training']['loss']['type'],
      '| epochs:', cfg['training']['epochs'], '| batch:', cfg['training']['batch_size'])
print('MODELS_ROOT    :', MODELS_ROOT)

In [ ]:
# === Muat hasil PREPROCESSING dari gdrive (deklarasi + panggil; BUKAN hitung ulang) ===
# Split sumber-aware IDENTIK Bagian 14.1 (seed=42): val/test = Papua-only holdout;
# transfer + augmentasi offline -> train. Class weights dari distribusi Bagian 14.3.
import json
from forestwatch.constants import N_CLASSES, CLASS_NAMES
from forestwatch.data import build_dataloaders_from_files, list_patches, split_files
from forestwatch.training.metrics import median_frequency_weights

papua_files    = list_patches(PATCH_DIR)
transfer_files = list_patches(PATCHES_TRANSFER)
aug_files      = list_patches(AUGMENTED_PATCHES)
assert papua_files, (
    f"Tidak ada patch di {PATCH_DIR}. Pastikan Bagian 1-14 (notebook utama) sudah "
    "dijalankan & Google Drive sudah selesai sync."
)

train_p, val_p, test_p = split_files(papua_files, train_ratio=0.8, val_ratio=0.1, seed=42)
final_train_files = list(train_p) + list(transfer_files) + list(aug_files)
print(f"train={len(final_train_files)} (papua={len(train_p)}+transfer={len(transfer_files)}"
      f"+aug={len(aug_files)}), val={len(val_p)}, test={len(test_p)} (Papua holdout)")

_post_aug = DIST_DIR / 'distribution_post_augmentation_on_target.json'
assert _post_aug.exists(), (
    f"{_post_aug} belum ada — jalankan Bagian 14.3 di notebook utama dulu "
    "(recompute distribusi TRAIN FINAL + median-frequency weights)."
)
dist_final = {int(k): int(v) for k, v in json.load(open(_post_aug))['counts'].items()}
class_weights = median_frequency_weights(dist_final, n_classes=N_CLASSES)
print('class_weights (median-freq):', [round(float(w), 3) for w in class_weights])

In [ ]:
# === Worker tuning (dinamis: optimal di lab multi-core, aman di Colab 2-vCPU) ===
import os
N_WORKERS = max(1, min(8, (os.cpu_count() or 2) - 1))
print(f"os.cpu_count()={os.cpu_count()} -> num_workers={N_WORKERS}; "
      "persistent_workers=True, pin_memory=True (di-set saat build DataLoader).")

## Bagian 15.5 — Banding 3 Model & Tetapkan Pemenang

In [ ]:
# === Bagian 15.5 — Baca summary.json ke-3 model + tabel + bar chart ===
import json
import matplotlib.pyplot as plt
from forestwatch.utils.io import save_json, load_json

EXPECTED = ['model_1_attention_unet', 'model_2_deeplabv3plus', 'model_3_unetpp']
rows = []
for key in EXPECTED:
    p = MODELS_ROOT / key / 'summary.json'
    if p.exists():
        rows.append(json.load(open(p)))
    else:
        print(f"[belum ada] {p} — model '{key}' mungkin belum selesai / belum sync.")
assert rows, "Belum ada summary.json satu pun. Jalankan notebook training dulu."

print(f"\n{'model':<26}{'arch':<15}{'val mIoU':>9}{'test mIoU':>10}{'OA':>8}{'kappa':>8}{'param':>13}{'menit':>7}")
print('-' * 96)
for r in sorted(rows, key=lambda x: x['test_mean_iou'], reverse=True):
    print(f"{r['model_key']:<26}{r['architecture']:<15}{r['best_val_iou']:>9.4f}{r['test_mean_iou']:>10.4f}"
          f"{r['test_overall_accuracy']*100:>7.1f}%{r['test_kappa']:>8.4f}{r['n_parameters']:>13,}{r['train_minutes']:>7}")

fig, ax = plt.subplots(figsize=(8, 4))
names = [r['model_key'] for r in rows]; mious = [r['test_mean_iou'] for r in rows]
ax.bar(names, mious, color=['#2c7fb8', '#7fcdbb', '#c7e9b4'][:len(rows)])
for i, v in enumerate(mious):
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom')
ax.set_ylabel('Test mIoU (Papua holdout)'); ax.set_ylim(0, 1)
ax.set_title('Perbandingan 3 Model — Test mIoU')
plt.xticks(rotation=12, ha='right'); fig.tight_layout()
fig.savefig(MODELS_ROOT / 'comparison.png', dpi=120, bbox_inches='tight'); plt.show()
save_json({'models': rows}, MODELS_ROOT / 'comparison.json')
print('Disimpan:', MODELS_ROOT / 'comparison.png', '+ comparison.json')

In [ ]:
# === Tetapkan pemenang (test mIoU tertinggi) + promosikan ke lokasi kanonik ===
import shutil, torch
from forestwatch.model.architecture import build_unet, count_parameters

winner = max(rows, key=lambda x: x['test_mean_iou'])
WIN_DIR = MODELS_ROOT / winner['model_key']
print(f"PEMENANG: {winner['model_key']} ({winner['architecture']}) — test mIoU={winner['test_mean_iou']:.4f}")
save_json(winner, MODELS_ROOT / 'winner.json')

# Bangun model pemenang + load bobot (utk inferensi).
model = build_unet(in_channels=cfg['model']['in_channels'], classes=cfg['model']['classes'],
                   encoder_weights=None, architecture=winner['architecture'],
                   encoder_name=winner['encoder_name'])
model.load_state_dict(torch.load(WIN_DIR / 'best_model.pt', map_location='cpu'))
model.eval()

# Promosikan artefak pemenang -> lokasi kanonik (supaya hilir/Orang-2 tak berubah).
CKPT_PATH = PATCH_DIR / 'best_model.pt'
OUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(WIN_DIR / 'best_model.pt', CKPT_PATH)
shutil.copy(WIN_DIR / 'metrics.json', OUT_DIR / 'metrics.json')
shutil.copy(WIN_DIR / 'model.onnx',  OUT_DIR / 'model.onnx')
shutil.copy(WIN_DIR / 'confusion_matrix.png', OUT_DIR / 'confusion_matrix.png')
# selaraskan cfg ke arsitektur pemenang (utk model card).
cfg['model']['architecture'] = winner['architecture']
cfg['model']['encoder_name'] = winner['encoder_name']
print('Dipromosikan ->', CKPT_PATH)
print('             ->', OUT_DIR / 'metrics.json', '| model.onnx | confusion_matrix.png')

## Bagian 16 — Inferensi T1 + T2 (model pemenang)

In [ ]:
# === Bagian 16 — Inferensi T1 + T2 (model pemenang) ===
from forestwatch.inference.tile_inference import infer_tiles_folder
model.eval()
_stride = cfg['inference'].get('stride', cfg['inference']['patch_size'])
_tta = cfg['inference'].get('tta', False)
print(f"Inferensi: patch={cfg['inference']['patch_size']}, stride={_stride}, tta={_tta}")

infer_tiles_folder(tile_dir=TILES_T2, out_dir=MASK_DIR, model=model, prefix='mask_t2_',
                   patch_size=cfg['inference']['patch_size'], stride=_stride, tta=_tta)
infer_tiles_folder(tile_dir=TILES_T1, out_dir=MASK_DIR, model=model, prefix='mask_t1_',
                   patch_size=cfg['inference']['patch_size'], stride=_stride, tta=_tta)
print('Inferensi T1 + T2 selesai ->', MASK_DIR)

## Bagian 17 — Generate 7 File Kontrak + Validasi + Ringkasan

In [ ]:
# === Bagian 17 — Generate 7 file kontrak (dari model pemenang) ===
from forestwatch.outputs.orchestrator import generate_all_outputs

metrics = load_json(OUT_DIR / 'metrics.json')
paths = generate_all_outputs(
    mask_dir=MASK_DIR, out_dir=OUT_DIR,
    period_from=cfg['periods']['t1'], period_to=cfg['periods']['t2'],
    metrics=metrics, onnx_src=OUT_DIR / 'model.onnx',
    min_area_ha=cfg['change_detection']['min_area_ha'],
    model_card_kwargs=dict(epochs=cfg['training']['epochs'], batch_size=cfg['training']['batch_size'],
                           n_parameters=count_parameters(model)),
)
for k, v in paths.items():
    print(f"  {k:<25} {v}")

In [ ]:
# === Validasi schema 7 file - WAJIB sebelum kirim ke Orang 2 ===
from forestwatch.validation.schema import validate_outputs_dir

report = validate_outputs_dir(OUT_DIR)
print(report.render())
assert report.ok, 'Validasi gagal - perbaiki sebelum hands-off ke Orang 2.'
print(f"\nOK Pipeline lengkap. Model final = {winner['model_key']} ({winner['architecture']}), "
      f"test mIoU={winner['test_mean_iou']:.4f}.")

In [ ]:
# === Ringkasan hasil (dari statistics.json pemenang) ===
stats = load_json(OUT_DIR / 'statistics.json')
print('=' * 60)
print('RINGKASAN HASIL - ForestWatch Papua')
print('=' * 60)
print(f"Model final          : {winner['model_key']} ({winner['architecture']})")
print(f"Periode pembanding   : {stats['period_from']} -> {stats['period_to']}")
print(f"Total deforestasi    : {stats['total_deforestation_ha']:>12,.1f} ha")
print(f"Jumlah hotspot       : {stats['n_hotspots']:>12,}")
print()
print('Per jenis transisi:')
for tname, ha in stats['per_transition_ha'].items():
    print(f"  {tname:<28} {ha:>12,.1f} ha")
print()
print('Akurasi model (pemenang):')
mm = stats['model_metrics']
print(f"  Overall Accuracy            {mm['overall_accuracy']*100:>11.2f}%")
print(f"  Mean IoU                    {mm['mean_iou']:>11.4f}")
if 'kappa' in mm:
    print(f"  Cohen's Kappa               {mm['kappa']:>11.4f}")
for row in mm['per_class']:
    print(f"  IoU {row['class']:<24} {row['iou']:>11.4f}")